# TP 5: PCA and Anomaly Detection

## 📝 Exercise 1: PCA on digits

In this exercise, you will apply **Principal Component Analysis (PCA)** to the `sklearn digits` dataset to understand dimensionality reduction and anomaly detection. You will analyze how data can be effectively compressed while preserving information.

### Part 0: Data Loading and Exploration

- Load the digits dataset from sklearn
- Explore the shape and characteristics of the data
- Visualize a sample of digits

In [ ]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load the digits dataset
digits = load_digits()
X = digits.data
y = digits.target

print("Dataset shape:", X.shape)
print("Target classes:", np.unique(y))
print("Number of features:", X.shape[1])
print("Number of samples:", X.shape[0])

for i in range(1,10):
    plt.subplot(330+i)
    plt.imshow(digits.images[i-1],cmap='Greys')

plt.show() 

### Part 1: Implement PCA for 2D Projection

Apply PCA to reduce the digit images from 64 dimensions to 2 dimensions. Then create a visualization showing how the different digit classes are distributed in this 2D space.

1. Create a PCA model with 2 principal components

2. Transform the data to the 2D space using your model

3. Create a scatter plot where:
    - The x-axis is the first principal component
    - The y-axis is the second principal component
    - Each digit class (0-9) has a different color
    - Add a colorbar to show which color represents which digit

In [ ]:
## TODO:
# 1)
pca = PCA(n_components=2)

In [ ]:
# 2)
X_pca = pca.fit_transform(X)

In [ ]:
X_pca.shape

In [ ]:
# 3)
plt.figure(figsize=(8,6))
scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
    cmap='tab10',
    s=15,
    alpha=0.7
)

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Projection of Digits Dataset (2D)')
plt.colorbar(scatter, label='Digit label')
plt.grid(True)
plt.show()

### Part 2: K-means Clustering with 10 Clusters

In the 2D PCA space you created in Exercise 1, apply K-means clustering to create 10 clusters (one for each digit). Then evaluate how well the clusters match the true digit labels.

1. Apply K-means with k=10 clusters to your 2D PCA data:
    - You may need to try multiple initializations to get exactly 10 non-empty clusters

2. Assign cluster labels by finding which digit class is most common in each cluster:
    - For each cluster, find the majority digit label
    - Assign this label to all points in that cluster

3. Evaluate the clustering using three metrics:
    - Precision: For each true digit class, what fraction of its samples were clustered correctly?
    - Recall: For each cluster, what fraction of its samples actually belong to the same digit class?
    - Gini-index: What percentage of all samples are in the "correct" cluster for their digit class?

In [ ]:
## TODO:
# 1)
kmeans = KMeans(n_clusters=10, n_init=20, random_state=42)
cluster_labels = kmeans.fit_predict(X_pca)

In [ ]:
# 2)
cluster_to_digit = {}

for cluster in range(10):
    indices = np.where(cluster_labels == cluster)[0]
    true_labels = y[indices]
    
    most_common_digit = Counter(true_labels).most_common(1)[0][0]
    cluster_to_digit[cluster] = most_common_digit

In [ ]:
y_pred = np.array([cluster_to_digit[c] for c in cluster_labels])

In [ ]:
# 3)
precision = precision_score(y, y_pred, average='macro')
precision

In [ ]:
recall = recall_score(y, y_pred, average='macro')
recall

In [ ]:
gini_index = np.mean(y_pred == y)
gini_index

### Part 3: 3D PCA Projection

Extend your analysis from 2D to 3D. With an extra dimension, you should capture more variance and see better separation of digit classes.

1. Create a 3D PCA model with 3 principal components

2. Transform the data to 3D space

3. Create a 3D scatter plot where:
    - Each digit class has a different color

In [ ]:
## TODO:
# 1)
pca_3d = PCA(n_components=3)

In [ ]:
# 2)
X_pca_3d = pca_3d.fit_transform(X)

In [ ]:
# 3)
fig = plt.figure(figsize=(9,7))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    X_pca_3d[:, 0],
    X_pca_3d[:, 1],
    X_pca_3d[:, 2],
    c=y,
    cmap='tab10',
    s=15,
    alpha=0.7
)

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
ax.set_title('3D PCA Projection of Digits Dataset')

fig.colorbar(scatter, ax=ax, label='Digit label')
plt.show()

### Part 4: Optimal Dimensionality Selection

**How many dimensions do we actually need?**

Too few and we lose important information, too many and we might overfit or negate the benefits of dimensionality reduction. We need to find the sweet spot.

1. Run PCA with increasing numbers of components:
    - For each k from 1 to 63:
        - Fit a PCA model with k components
        - Calculate the explained variance ratio

2. Create a plot showing:
    - X-axis: component number (k = 1 to 63)
    - Y-axis: explained variance ratio

In [ ]:
## TODO:
explained_variance = []

for k in range(1, 64):
    pca = PCA(n_components=k)
    pca.fit(X)
    explained_variance.append(np.sum(pca.explained_variance_ratio_))

In [ ]:
for i in range(0,63):
    print(explained_variance[i])

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1, 64), explained_variance, marker='o')
plt.xlabel('Number of Principal Components (k)')
plt.ylabel('Cumulative Explained Variance')
plt.title('Explained Variance vs Number of PCA Components')
plt.grid(True)
plt.show()

## 📝 Exercise 2: Anomaly Detection in Facebook Spatial Likes

In this exercise, you will apply the low-dimensional phenomenon to detect anomalous users in a real Facebook dataset. The dataset contains information about users' likes across different content categories.

- Dataset Description:
    - 9,000 users
    - 210 content categories (different categories of pages on Facebook)
    - 6 months of data

- Each entry represents the number of likes a user gave to pages in that category

- Data Format:
    - Rows = users
    - Columns = content categories
    - Matrix shape: 9000 × 210

### Part 0: Data Loading and Exploration

- Load the data `spatial_data.txt` and extract the content categories (exclude the first column which is user IDs).

In [ ]:
# Load data
data = np.loadtxt('spatial_data.txt')
FBSpatial = data[:,1:]  # Remove user ID column
print(f"Data shape: {FBSpatial.shape}")
print(f"Number of users: {FBSpatial.shape[0]}")
print(f"Number of categories: {FBSpatial.shape[1]}")

- Check the total number of likes for each user (the row sums).

In [ ]:
FBSnorm = np.linalg.norm(FBSpatial,axis=1,ord=1)
plt.plot(FBSnorm)
plt.title('Number of Likes Per User')
plt.xlabel('Users')

In many real-world datasets, most "normal" observations lie approximately in a low-dimensional subspace. This is called the low-dimensional phenomenon. What does this mean?

- Normal users might have similar like patterns across 210 categories
- These patterns can be summarized by just 20-25 principal components
- Anomalous users break this pattern - they have unusual preference profiles 

Let's check whether the low dimensional phenomenon holds.

In [ ]:
u,s,vt = np.linalg.svd(FBSpatial,full_matrices=False)
plt.plot(s/np.linalg.norm(FBSpatial))
plt.title('Singular Values of Spatial Like Matrix')

### Part 1: Build the Normal Model and Extract Anomalies

Separate the portion of the data lying in the normal space from the amonalous space.

1. Compute SVD of FBSpatial

2. Use first 25 columns (which captures ~80-85% of the variance) to project data onto normal subspace

3. Calculate residuals as anomalous component

In [ ]:
## TODO:
# 1) 
u, s, vt = np.linalg.svd(FBSpatial, full_matrices=False)

In [ ]:
# 2)
k = 25

Uk = u[:, :k]
Sk = np.diag(s[:k])
Vk = vt[:k, :]

In [ ]:
FB_normal = Uk @ Sk @ Vk

In [ ]:
# 3)
FB_anomalous = FBSpatial - FB_normal

### Part 2: Identify Top 30 Anomalous Users

Find the 30 users with largest anomaly scores and visualize their position relative to overall user activity.

In [ ]:
## TODO:
anomaly_score = np.linalg.norm(FB_anomalous, axis=1)

In [ ]:
anomaly_score

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(anomaly_score)
plt.xlabel('Users')
plt.ylabel('Anomaly Score (Residual Norm)')
plt.title('Anomaly Scores from PCA Residuals')
plt.show()

In [ ]:
top_30_idx = np.argsort(anomaly_score)[-30:]

In [ ]:
top_30_scores = anomaly_score[top_30_idx]

In [ ]:
activity = np.linalg.norm(FBSpatial, axis=1, ord=1)

In [ ]:
plt.figure(figsize=(8,6))

# Tous les utilisateurs
plt.scatter(activity, anomaly_score, alpha=0.3, label='Normal users')

# Top 30 anomalies
plt.scatter(
    activity[top_30_idx],
    anomaly_score[top_30_idx],
    color='red',
    s=60,
    label='Top 30 anomalies'
)

plt.xlabel('Total number of likes (L1 norm)')
plt.ylabel('Anomaly score (residual norm)')
plt.title('Top 30 Anomalous Users in Facebook Spatial Likes')
plt.legend()
plt.grid(True)
plt.show()

### Part 3: Visualize Patterns of Anomalous Users

Plot the like patterns (across the 210 categories) for 9 of the top anomalous users.

In [ ]:
## TODO:
top_9_idx = np.argsort(anomaly_score)[-9:]

In [ ]:
anomalous_profiles = FBSpatial[top_9_idx]

In [ ]:
plt.figure(figsize=(15,10))

for i, idx in enumerate(top_9_idx):
    plt.subplot(3, 3, i + 1)
    plt.plot(FBSpatial[idx])
    plt.title(f'User {idx}')
    plt.xlabel('Category index')
    plt.ylabel('Number of likes')

plt.suptitle('Like Patterns of Top 9 Anomalous Users', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
